# 01 – Parse & Clean NASA HTTP Logs

This notebook:
1. Loads the sample log file shipped with the repo (`data/sample_logs.txt`).
2. Parses it into a structured DataFrame using `pipeline.log_parser`.
3. Applies basic cleaning.
4. Saves the result to `data/processed/parsed_logs.parquet`.

In [ ]:
import sys, pathlib
# Ensure repo root is on the path when running from notebooks/
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))


In [ ]:
import pandas as pd
from pipeline.log_parser import parse_log_file

LOG_FILE = ROOT / "data" / "sample_logs.txt"
print(f"Parsing {LOG_FILE} …")
df = parse_log_file(str(LOG_FILE))
print(f"Parsed {len(df):,} log entries.")
df.head()


In [ ]:
# ── Basic cleaning ────────────────────────────────────────────────────────────
# 1. Drop rows with a null timestamp (malformed lines)
before = len(df)
df = df.dropna(subset=["timestamp"])
print(f"Dropped {before - len(df)} rows with null timestamps.")

# 2. Filter out HEAD/OPTIONS noise
df = df[~df["method"].isin(["HEAD", "OPTIONS"])].copy()
print(f"Rows after method filter: {len(df):,}")

# 3. Normalise endpoint (strip query string)
df["endpoint"] = df["endpoint"].str.split("?").str[0]

# 4. Clip absurdly large byte values (> 10 MB) to NaN then fill with median
BYTES_MAX = 10_000_000
df.loc[df["bytes_sent"] > BYTES_MAX, "bytes_sent"] = df["bytes_sent"].median()

df.dtypes


In [ ]:
# ── Save processed dataset ────────────────────────────────────────────────────
OUT = ROOT / "data" / "processed" / "parsed_logs.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(str(OUT), index=False)
print(f"Saved → {OUT}  ({len(df):,} rows)")


In [ ]:
# ── Quick sanity check ────────────────────────────────────────────────────────
print("Columns:", df.columns.tolist())
print("\nStatus code distribution:")
print(df["status_code"].value_counts().head(10))
print("\nTop 5 IPs by request count:")
print(df["ip_address"].value_counts().head(5))
